<a href="https://colab.research.google.com/github/raw-fun/Colab-Script/blob/main/Web_Scrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Pro-level setup (Gradio ebong Playwright installation)
!pip install -q playwright pandas gradio nest_asyncio
!playwright install chromium
!playwright install-deps chromium

import asyncio
import re
import random
import pandas as pd
import nest_asyncio
from playwright.async_api import async_playwright
from urllib.parse import urljoin, urlparse

# Colab e Gradio ebong Asyncio ekshathe chalanor jonno eti joruri
nest_asyncio.apply()

print("🚀 Advanced Forensic Engine ebong Environment prostut.")

In [ ]:
import asyncio
import re
import random
import pandas as pd
from urllib.parse import urljoin, urlparse
from playwright.async_api import async_playwright

class DeepLinkForensicFinal:
    def __init__(self, root_url, max_pages=100):
        self.root_url = root_url.rstrip('/')
        self.max_pages = max_pages
        self.nodes = []
        self.visited = set()
        self.queue = [root_url]
        self.total_discovered = 1
        self.is_running = True

    def is_valid_page(self, url):
        """Webpage ebong file er moddhe parthokko kore filter korbe"""
        parsed = urlparse(url)
        # 1. Query param bad diye path ber kora
        clean_path = parsed.path.lower()

        # 2. Exclude static/document files
        excluded_ext = ('.css', '.js', '.ico', '.png', '.jpg', '.jpeg', '.svg', '.gif', '.pdf',
                        '.woff', '.woff2', '.zip', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx')

        if clean_path.endswith(excluded_ext):
            return False

        # 3. Skip internal file directories (to avoid 404s)
        if any(dir_name in clean_path for dir_name in ["/sites/default/files/", "/uploads/", "/assets/"]):
            return False

        # 4. Domain restriction (Stay within target domain)
        target_domain = urlparse(self.root_url).netloc
        return parsed.netloc == target_domain

    async def extract_links(self, page, current_url):
        """Page theke shob unique internal link ber korbe"""
        try:
            content = await page.content()
            # Advanced regex for href links
            links = re.findall(r'href=[\'"]?([^\'" >]+)', content)
            added = 0
            for link in set(links):
                full_url = urljoin(current_url, link).split('#')[0].split('?')[0].rstrip('/')

                if full_url not in self.visited and full_url not in self.queue and self.is_valid_page(full_url):
                    if len(self.visited) + len(self.queue) < self.max_pages + 500:
                        self.queue.append(full_url)
                        self.total_discovered += 1
                        added += 1
            return added
        except:
            return 0

    async def run(self):
        """Async generator for live dashboard updates"""
        logs = []
        def log_msg(msg, append=True):
            if append: logs.append(msg)
            else: logs[-1] = msg
            return "\n".join(logs[-10:]) # Keep last 10 lines for UI clarity

        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True, args=["--disable-blink-features=AutomationControlled"])
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
            )

            yield log_msg(f"🚀 Mission Started: {self.root_url}"), pd.DataFrame(), None, f"Found: {self.total_discovered} | Queue: {len(self.queue)}"

            while self.queue and len(self.nodes) < self.max_pages and self.is_running:
                url = self.queue.pop(0)
                if url in self.visited: continue
                self.visited.add(url)

                page = await context.new_page()
                # Bandwidth save: Block unnecessary resources
                await page.route("**/*.{css,js,png,jpg,ico,woff,svg,mp4}", lambda r: r.abort())

                success = False
                retries = 2
                while retries > 0 and not success:
                    try:
                        short_url = (url[:45] + '..') if len(url) > 45 else url
                        status_text = f"🔍 [{len(self.nodes)+1}] Visiting: {short_url}"
                        yield log_msg(status_text, True), pd.DataFrame(self.nodes), None, f"Visited: {len(self.nodes)} | Discovered: {self.total_discovered} | Queue: {len(self.queue)}"

                        response = await page.goto(url, wait_until="domcontentloaded", timeout=35000)
                        await asyncio.sleep(0.5) # Fast but safe delay
                        status = response.status if response else 200

                        self.nodes.append({"url": url, "status": status, "domain": urlparse(url).netloc})
                        yield log_msg(status_text + f" ✅ [{status}]", False), pd.DataFrame(self.nodes), None, f"Visited: {len(self.nodes)} | Discovered: {self.total_discovered} | Queue: {len(self.queue)}"

                        await self.extract_links(page, url)
                        success = True
                    except Exception:
                        retries -= 1
                        if retries == 0:
                            self.nodes.append({"url": url, "status": "Error/Timeout", "domain": urlparse(url).netloc})
                            yield log_msg(status_text + " ❌ Skipped", False), pd.DataFrame(self.nodes), None, f"Visited: {len(self.nodes)} | Queue: {len(self.queue)}"

                await page.close()

            await browser.close()
            df = pd.DataFrame(self.nodes)
            csv_path = "forensic_report.csv"
            if not df.empty: df.to_csv(csv_path, index=False)

            final_msg = "🏁 Scan Finished Successfully!" if not self.queue else "⚠️ Limit Reached - Scan Paused."
            yield log_msg(f"\n{final_msg}"), df, csv_path if not df.empty else None, f"Total Visited: {len(self.nodes)} | Remaining in Queue: {len(self.queue)}"

In [ ]:
import gradio as gr
import pandas as pd
import asyncio
import uvicorn.server

# Prevent infinite recursion when re-running cells in Colab
if not getattr(uvicorn.server, "_is_patched", False):
    if hasattr(uvicorn.server, "asyncio_run"):
        _original_uvicorn_run = uvicorn.server.asyncio_run
        def _patched_uvicorn_run(*args, **kwargs):
            kwargs.pop('loop_factory', None)
            return _original_uvicorn_run(*args, **kwargs)
        uvicorn.server.asyncio_run = _patched_uvicorn_run
        uvicorn.server._is_patched = True

async def launch_scanner(target_url, total_pages):
    # Basic validation
    if not target_url.startswith("http"):
        yield "❌ Error: Invalid URL format!", pd.DataFrame(), gr.update(interactive=False), "N/A"
        return

    scanner = DeepLinkForensicFinal(target_url, max_pages=total_pages)

    # Async Generator theke data niye UI smooth-vabe update kora
    async for log_txt, df, file, stats in scanner.run():
        if file:
            btn_update = gr.update(value=file, interactive=True, label="📥 Download Final CSV")
        else:
            btn_update = gr.update(value=None, interactive=False, label="⏳ Scanning in Progress...")

        yield log_txt, df, btn_update, stats

# UI Design with Premium Theme
theme = gr.themes.Soft(primary_hue="orange", secondary_hue="slate", font=[gr.themes.GoogleFont("Inter")])

# demo.launch() theke warning shorate Block er vitorei theme deya holo
with gr.Blocks(theme=theme, title="Link Forensic Pro v2.0") as demo:
    gr.HTML("<div style='text-align: center; padding: 10px;'><h1>🕵️ Advanced Link Forensic Dashboard</h1><p>Professional Website Audit & Deep Crawling Engine</p></div>")

    with gr.Column(variant="panel"):
        with gr.Row():
            url_input = gr.Textbox(label="Target Website URL", placeholder="https://example.gov.bd", scale=4)
            limit_input = gr.Slider(minimum=10, maximum=10000, value=200, step=50, label="Page Limit", scale=2)

        with gr.Row():
            btn_start = gr.Button("🚀 Start Forensic Scan", variant="primary", scale=3)
            csv_dl = gr.DownloadButton("📥 Download CSV Report", interactive=False, scale=1)

    stats_box = gr.Textbox(value="Status: Ready to Scan", label="Real-time Crawler Metrics", interactive=False, lines=1)

    with gr.Row():
        with gr.Column(variant="panel"):
            gr.Markdown("### 📡 Engine Live Logs")
            log_box = gr.Textbox(show_label=False, lines=6, interactive=False)

    with gr.Column(variant="panel"):
        gr.Markdown("### 📊 Extracted Links & Forensic Data")
        data_table = gr.Dataframe(headers=["url", "status", "domain"], interactive=False)

    btn_start.click(
        fn=launch_scanner,
        inputs=[url_input, limit_input],
        outputs=[log_box, data_table, csv_dl, stats_box]
    )

# debug=False set kora hoyeche jate Colab cell atke na thake
demo.launch(share=True, debug=False)